# ALDI scraper — estructura común final

Este notebook sigue el mismo flujo que DIA y Mercadona:

1. **Scraping** de las 20 búsquedas, con reintentos automáticos.
2. **RAW:** todas las apariciones de todos los términos se unen y se guardan en un único CSV.
3. **Procesado:** deduplicación, normalización y eliminación de filas sin precio o unidad comparable.
4. **PROCESSED:** un único CSV final limpio.

Archivos:
- `data/raw/aldi_products_raw.csv`
- `data/processed/aldi_products.csv`

In [188]:
import time
import re
from pathlib import Path
from datetime import datetime

import pandas as pd

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

HOME_URL = "https://www.aldi.es/"
WAIT = 15
MAX_RETRIES = 2

SEARCH_TERMS = [
    "leche", "huevos", "manzana", "plátano", "tomate", "arroz",
    "pasta", "pan", "pollo", "carne picada", "salmón", "yogur",
    "queso", "agua", "zumo", "atún", "garbanzos", "aceite",
    "detergente", "papel higiénico"
]

In [189]:
current = Path.cwd().resolve()
BASE_DIR = current.parent if current.name == "scrapers" else current

RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RAW_PATH = RAW_DIR / "aldi_products_raw.csv"
PROCESSED_PATH = PROCESSED_DIR / "aldi_products.csv"

print("RAW:", RAW_PATH)
print("PROCESSED:", PROCESSED_PATH)

RAW: C:\Users\Giuliana Galassi\OneDrive\Documentos\uc3m\4TO\WEB ANALYTICS\supermarket_project\data\raw\aldi_products_raw.csv
PROCESSED: C:\Users\Giuliana Galassi\OneDrive\Documentos\uc3m\4TO\WEB ANALYTICS\supermarket_project\data\processed\aldi_products.csv


In [190]:
def start_driver():
    options = Options()
    options.add_argument("--window-size=1920,1080")
    driver = webdriver.Chrome(options=options)
    driver.get(HOME_URL)
    return driver


def accept_cookies(driver):
    container = driver.execute_script("""
        function find(selector, root = document) {
            const el = root.querySelector(selector);
            if (el) return el;
            for (const node of root.querySelectorAll('*')) {
                if (node.shadowRoot) {
                    const found = find(selector, node.shadowRoot);
                    if (found) return found;
                }
            }
            return null;
        }
        return find('[data-testid="uc-container"]');
    """)

    if container is None:
        print("No apareció el popup de cookies.")
        return

    scroller = driver.execute_script("""
        return arguments[0].getRootNode()
            .querySelector('[data-testid="uc-vertical-scroller"]');
    """, container)

    if scroller is not None:
        driver.execute_script(
            "arguments[0].scrollTop = arguments[0].scrollHeight;",
            scroller
        )

    time.sleep(0.5)

    accept_button = driver.execute_script("""
        return arguments[0].getRootNode()
            .querySelector('[data-testid="uc-accept-all-button"]');
    """, container)

    if accept_button is None:
        raise RuntimeError("No se encontró el botón de aceptar cookies.")

    driver.execute_script("arguments[0].click();", accept_button)
    print("Cookies aceptadas.")


def select_peninsula_if_needed(driver):
    buttons = driver.find_elements(
        By.CSS_SELECTOR,
        'button[data-region-code="pen"]'
    )

    for button in buttons:
        if button.is_displayed() and button.is_enabled():
            button.click()
            time.sleep(0.5)
            print("Región seleccionada: Península")
            return


def go_to_home(driver):
    driver.get(HOME_URL)
    WebDriverWait(driver, WAIT).until(
        lambda d: d.execute_script("return document.readyState") == "complete"
    )
    time.sleep(1)
    select_peninsula_if_needed(driver)

In [191]:
def find_search_box(driver):
    selectors = [
        'input[type="search"]',
        'input[placeholder*="Buscar"]',
        'input[aria-label*="Buscar"]',
        'input[name*="search"]',
        'input[id*="search"]',
    ]

    def visible_input(_):
        for selector in selectors:
            for element in driver.find_elements(By.CSS_SELECTOR, selector):
                if element.is_displayed() and element.is_enabled():
                    return element
        return False

    return WebDriverWait(driver, WAIT).until(visible_input)


def search_product(driver, term):
    box = find_search_box(driver)
    box.click()
    box.send_keys(Keys.CONTROL, "a")
    box.send_keys(Keys.BACKSPACE)
    box.send_keys(term)
    box.send_keys(Keys.ENTER)

    WebDriverWait(driver, WAIT).until(
        lambda d: term.lower() in d.find_element(By.TAG_NAME, "body").text.lower()
    )

    WebDriverWait(driver, WAIT).until(
        lambda d: (
            len(d.find_elements(By.CSS_SELECTOR, ".product-tile")) > 0
            or "Productos (0)" in d.find_element(By.TAG_NAME, "body").text
        )
    )
    time.sleep(1)


def get_expected_product_count(driver):
    for tab in driver.find_elements(By.CSS_SELECTOR, '[data-testid="tab"]'):
        text = tab.text.strip()
        if text.startswith("Productos"):
            match = re.search(r"\((\d+)\)", text)
            if match:
                return int(match.group(1))
    return None


def get_loaded_product_count(driver):
    return len(driver.find_elements(By.CSS_SELECTOR, ".product-tile"))


def load_all_products(driver, expected_count, max_clicks=40):
    loaded = get_loaded_product_count(driver)
    clicks = 0
    print(f"Cargados inicialmente: {loaded}/{expected_count}")

    while loaded < expected_count:
        buttons = driver.find_elements(
            By.CSS_SELECTOR,
            '[data-testid="product-tile-grid-load-more-button"]'
        )
        buttons = [
            b for b in buttons
            if b.is_displayed() and b.is_enabled()
        ]

        if not buttons:
            raise RuntimeError(
                f"No aparece 'Mostrar más' con {loaded}/{expected_count} cargados."
            )

        button = buttons[0]
        driver.execute_script(
            "arguments[0].scrollIntoView({block:'center'});",
            button
        )

        previous = loaded

        try:
            button.click()
        except Exception:
            driver.execute_script("arguments[0].click();", button)

        clicks += 1

        WebDriverWait(driver, WAIT).until(
            lambda d: get_loaded_product_count(d) > previous
        )

        loaded = get_loaded_product_count(driver)
        print(f"Click {clicks}: {previous} -> {loaded}/{expected_count}")

        if clicks >= max_clicks and loaded < expected_count:
            raise RuntimeError("Se alcanzó el máximo de clicks.")

    return loaded, clicks

In [192]:
def extract_products(driver, searched_item):
    products = driver.execute_script("""
        function collectProducts(root = document, results = []) {
            const cards = root.querySelectorAll('.product-tile');

            for (const card of cards) {
                const brand = card.querySelector(
                    '.product-tile__content__upper__brand-name'
                );
                const name = card.querySelector(
                    '.product-tile__content__upper__product-name'
                );
                const price = card.querySelector(
                    '[data-testid$="-tag-current-price-amount"]'
                );
                const salesUnit = card.querySelector(
                    '[data-testid$="-tag-sales-unit"]'
                );
                const basePrice = card.querySelector(
                    '[data-testid$="-tag-base-price"]'
                );
                const link = card.querySelector('a.product-tile__action');

                results.push({
                    brand: brand ? brand.innerText.trim() : null,
                    name: name ? name.innerText.trim() : null,
                    price: price ? price.innerText.trim() : null,
                    quantity_raw: salesUnit ? salesUnit.innerText.trim() : null,
                    price_per_unit_raw: basePrice ? basePrice.innerText.trim() : null,
                    url: link ? link.href : null
                });
            }

            for (const el of root.querySelectorAll('*')) {
                if (el.shadowRoot) collectProducts(el.shadowRoot, results);
            }
            return results;
        }
        return collectProducts();
    """)

    df = pd.DataFrame(products)

    if df.empty:
        return df

    df["price"] = pd.to_numeric(df["price"], errors="coerce")
    df["product_id"] = df["url"].str.extract(
        r"-(\d+)\.html$", expand=False
    )
    df["supermarket"] = "ALDI"
    df["searched_item"] = searched_item
    df["scraped_at"] = datetime.now().isoformat(timespec="seconds")

    return df

In [193]:
driver = start_driver()
time.sleep(2)
accept_cookies(driver)
time.sleep(1)
select_peninsula_if_needed(driver)

all_results = []
summary = []

for i, term in enumerate(SEARCH_TERMS, start=1):
    success = False
    started = time.monotonic()

    for attempt in range(1, MAX_RETRIES + 2):
        print("\n" + "=" * 70)
        print(f"[{i}/{len(SEARCH_TERMS)}] {term} · intento {attempt}")

        try:
            go_to_home(driver)
            search_product(driver, term)

            expected = get_expected_product_count(driver)
            if expected is None:
                raise RuntimeError("No se pudo leer el contador de productos.")

            loaded, clicks = load_all_products(driver, expected)
            df_term = extract_products(driver, term)
            extracted = len(df_term)

            if expected != loaded or expected != extracted:
                raise RuntimeError(
                    f"Esperados {expected}, cargados {loaded}, extraídos {extracted}."
                )

            elapsed = round(time.monotonic() - started, 2)
            all_results.append(df_term)

            summary.append({
                "searched_item": term,
                "expected_products": expected,
                "loaded_products": loaded,
                "extracted_products": extracted,
                "load_more_clicks": clicks,
                "attempts": attempt,
                "status": "OK",
                "seconds": elapsed,
            })

            print("Estado: OK")
            success = True
            break

        except Exception as exc:
            print(f"Intento {attempt} fallido: {type(exc).__name__}: {exc}")
            if attempt < MAX_RETRIES + 1:
                time.sleep(2)

    if not success:
        summary.append({
            "searched_item": term,
            "expected_products": None,
            "loaded_products": None,
            "extracted_products": 0,
            "load_more_clicks": None,
            "attempts": MAX_RETRIES + 1,
            "status": "ERROR",
            "seconds": round(time.monotonic() - started, 2),
        })

df_summary = pd.DataFrame(summary)
display(df_summary)

Cookies aceptadas.
Región seleccionada: Península

[1/20] leche · intento 1
Cargados inicialmente: 9/99
Click 1: 9 -> 18/99
Click 2: 18 -> 27/99
Click 3: 27 -> 36/99
Click 4: 36 -> 45/99
Click 5: 45 -> 54/99
Click 6: 54 -> 63/99
Click 7: 63 -> 72/99
Click 8: 72 -> 81/99
Click 9: 81 -> 90/99
Click 10: 90 -> 99/99
Estado: OK

[2/20] huevos · intento 1
Cargados inicialmente: 9/33
Click 1: 9 -> 18/33
Click 2: 18 -> 27/33
Click 3: 27 -> 33/33
Estado: OK

[3/20] manzana · intento 1
Cargados inicialmente: 9/19
Click 1: 9 -> 18/19
Click 2: 18 -> 19/19
Estado: OK

[4/20] plátano · intento 1
Cargados inicialmente: 9/13
Click 1: 9 -> 13/13
Estado: OK

[5/20] tomate · intento 1
Cargados inicialmente: 9/50
Click 1: 9 -> 18/50
Click 2: 18 -> 27/50
Click 3: 27 -> 36/50
Click 4: 36 -> 45/50
Click 5: 45 -> 50/50
Estado: OK

[6/20] arroz · intento 1
Cargados inicialmente: 9/26
Click 1: 9 -> 18/26
Click 2: 18 -> 26/26
Estado: OK

[7/20] pasta · intento 1
Cargados inicialmente: 9/105
Click 1: 9 -> 18/105


,searched_item,expected_products,loaded_products,extracted_products,load_more_clicks,attempts,status,seconds
0,leche,99,99,99,10,1,OK,23.42
1,huevos,33,33,33,3,1,OK,11.08
2,manzana,19,19,19,2,1,OK,8.50
3,plátano,13,13,13,1,1,OK,6.91
4,tomate,50,50,50,5,1,OK,14.20
5,arroz,26,26,26,2,1,OK,8.50
6,pasta,105,105,105,11,1,OK,26.12
7,pan,117,117,117,12,1,OK,28.89
8,pollo,81,81,81,8,1,OK,24.00
9,carne picada,2,2,2,0,1,OK,5.62


## RAW — unir todas las búsquedas y guardar un único archivo

In [194]:
if not all_results:
    raise RuntimeError("No se extrajo ninguna búsqueda.")

df_aldi_raw = pd.concat(all_results, ignore_index=True)

df_aldi_raw.to_csv(
    RAW_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Filas RAW:", len(df_aldi_raw))
print("IDs distintos:", df_aldi_raw["product_id"].nunique())
print("RAW guardado:", RAW_PATH)

Filas RAW: 935
IDs distintos: 814
RAW guardado: C:\Users\Giuliana Galassi\OneDrive\Documentos\uc3m\4TO\WEB ANALYTICS\supermarket_project\data\raw\aldi_products_raw.csv


## PROCESSED — deduplicar, normalizar y eliminar filas incompletas

In [195]:
def normalize_base_price(raw_value):
    if pd.isna(raw_value):
        return None, None

    text = str(raw_value).lower().strip().replace(",", ".")

    match = re.search(
        r"^(kg|l|100-ml|100 ml|100-g|100 g|100-gr|100 gr|"
        r"unidad|lavado|capsula|cápsula|bolsita|pastilla|"
        r"panuelo|pañuelo|toallita|m-\(metro\)|metro)"
        r"\s*=\s*(\d+(?:\.\d+)?)",
        text
    )

    if not match:
        return None, None

    raw_unit = match.group(1)
    value = float(match.group(2))

    if raw_unit == "kg":
        return value, "KILO"
    if raw_unit == "l":
        return value, "LITRO"
    if raw_unit in {"100-ml", "100 ml"}:
        return value * 10, "LITRO"
    if raw_unit in {"100-g", "100 g", "100-gr", "100 gr"}:
        return value * 10, "KILO"

    mapping = {
        "unidad": "UNIDAD",
        "lavado": "LAVADO",
        "capsula": "CAPSULA",
        "cápsula": "CAPSULA",
        "bolsita": "BOLSITA",
        "pastilla": "PASTILLA",
        "panuelo": "PANUELO",
        "pañuelo": "PANUELO",
        "toallita": "TOALLITA",
        "m-(metro)": "METRO",
        "metro": "METRO",
    }

    return value, mapping.get(raw_unit)


def calculate_missing_unit_price(row):
    if pd.notna(row["price_per_unit"]):
        return row["price_per_unit"], row["unit"]

    if pd.isna(row["price"]):
        return None, None

    price = float(row["price"])

    if pd.notna(row["quantity_raw"]):
        text = str(row["quantity_raw"]).lower().strip()

        if "precio kg" in text:
            return price, "KILO"

        match = re.search(
            r"(\d+(?:[.,]\d+)?)\s*(kg|g|gr|l|ml)",
            text
        )

        if match:
            quantity = float(match.group(1).replace(",", "."))
            raw_unit = match.group(2)

            if quantity == 0:
                return None, None
            if raw_unit == "kg":
                return price / quantity, "KILO"
            if raw_unit in {"g", "gr"}:
                return price / (quantity / 1000), "KILO"
            if raw_unit == "l":
                return price / quantity, "LITRO"
            if raw_unit == "ml":
                return price / (quantity / 1000), "LITRO"

        pack_match = re.search(
            r"pack de\s*(\d+(?:[.,]\d+)?)\s*"
            r"(cápsulas?|capsulas?|bolsitas?|pastillas?|"
            r"pañuelos?|panuelos?|toallitas?|monodosis)",
            text
        )

        if pack_match:
            quantity = float(pack_match.group(1).replace(",", "."))
            raw_unit = pack_match.group(2)

            unit_map = {
                "cápsula": "CAPSULA", "cápsulas": "CAPSULA",
                "capsula": "CAPSULA", "capsulas": "CAPSULA",
                "bolsita": "BOLSITA", "bolsitas": "BOLSITA",
                "pastilla": "PASTILLA", "pastillas": "PASTILLA",
                "pañuelo": "PANUELO", "pañuelos": "PANUELO",
                "panuelo": "PANUELO", "panuelos": "PANUELO",
                "toallita": "TOALLITA", "toallitas": "TOALLITA",
                "monodosis": "MONODOSIS",
            }
            return price / quantity, unit_map[raw_unit]

        metre_match = re.search(
            r"(?:pack de\s*)?(\d+(?:[.,]\d+)?)\s*metros?",
            text
        )

        if metre_match:
            quantity = float(metre_match.group(1).replace(",", "."))
            return price / quantity, "METRO"

        if text == "unidad":
            return price, "UNIDAD"

    # Caso observado en ALDI: el HTML muestra 200 y "= 14.95"
    # sin conservar la "g", pero el precio base publicado es 14.95 €/kg.
    if (
        row["name"] == "Caprichos crujientes con pipas y pasas"
        and str(row["price_per_unit_raw"]).strip() == "= 14.95"
    ):
        return 14.95, "KILO"

    return None, None

In [196]:
df_aldi_processed = (
    df_aldi_raw
    .drop_duplicates(subset=["product_id"])
    .reset_index(drop=True)
    .copy()
)

normalized = df_aldi_processed["price_per_unit_raw"].apply(
    normalize_base_price
)

df_aldi_processed["price_per_unit"] = normalized.apply(lambda x: x[0])
df_aldi_processed["unit"] = normalized.apply(lambda x: x[1])

missing = df_aldi_processed["price_per_unit"].isna()

calculated = df_aldi_processed.loc[missing].apply(
    calculate_missing_unit_price,
    axis=1
)

df_aldi_processed.loc[missing, "price_per_unit"] = (
    calculated.apply(lambda x: x[0])
)
df_aldi_processed.loc[missing, "unit"] = (
    calculated.apply(lambda x: x[1])
)

df_aldi_processed["price_per_unit"] = pd.to_numeric(
    df_aldi_processed["price_per_unit"],
    errors="coerce"
).round(2)

before_drop = len(df_aldi_processed)

df_aldi_processed = (
    df_aldi_processed[
        df_aldi_processed["price"].notna()
        & df_aldi_processed["price_per_unit"].notna()
        & df_aldi_processed["unit"].notna()
    ]
    .reset_index(drop=True)
)

print("Únicos antes de eliminar incompletos:", before_drop)
print("Productos finales:", len(df_aldi_processed))
print("Eliminados por precio/unidad incompletos:", before_drop - len(df_aldi_processed))
print("\nUnidades:")
print(df_aldi_processed["unit"].value_counts(dropna=False))

Únicos antes de eliminar incompletos: 814
Productos finales: 773
Eliminados por precio/unidad incompletos: 41

Unidades:
unit
KILO         596
LITRO        119
UNIDAD        27
LAVADO        13
METRO          8
BOLSITA        4
CAPSULA        2
PASTILLA       1
MONODOSIS      1
PANUELO        1
TOALLITA       1
Name: count, dtype: int64


In [197]:
df_aldi_processed.to_csv(
    PROCESSED_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("PROCESSED guardado:", PROCESSED_PATH)

PROCESSED guardado: C:\Users\Giuliana Galassi\OneDrive\Documentos\uc3m\4TO\WEB ANALYTICS\supermarket_project\data\processed\aldi_products.csv


In [198]:
driver.quit()
print("Driver cerrado.")

Driver cerrado.
